# Temporal CycleGAN Inference

This notebook demonstrates how to load a trained temporal CycleGAN and generate sequences for new patients.

In [3]:
import sys
import os
os.chdir('/home/andrea/Desktop/Sarcoma-DT')
import json

from pathlib import Path
from typing import List

import torch
import pandas as pd
from torch.utils.data import DataLoader

from Models.models.GANs import LSTMGenerator
from Models.models.decoder import Decoder
from Models.models.utils import MetadataHandler
from Models.utils.helpers import MongoExtractor
from Models.dataset.tabular_dataset import TabularDatasetPID

def load_temporal_cycle_gan(meta_path: Path, device: torch.device):
    """Load Gx generator from saved metadata."""
    with open(meta_path, "r", encoding="utf-8") as f:
        meta = json.load(f)

    params = meta.get("model_params", {})
    embedding_dim = params.get("embedding_dim", 192)
    hidden_dim = params.get("hidden_dim", 128)
    num_layers = params.get("num_layers", 1)

    gx = LSTMGenerator(
        input_dim=embedding_dim,
        cond_input_dim=3,
        hidden_dim=hidden_dim,
        output_dim=embedding_dim,
        num_layers=num_layers,
    ).to(device)
    gx.load_state_dict(torch.load(meta["Gx_path"], map_location=device))
    gx.eval()
    return gx, meta

def load_decoder(meta_file: str, decoder_path: str, device: torch.device):
    """Load decoder network using parameters stored in metadata."""
    with open(meta_file, "r", encoding="utf-8") as f:
        meta = json.load(f)
    params = meta.get("model_params", {})
    hidden_dim = params.get("hidden_dim")
    output_dim = params.get("input_dim")
    dec = Decoder(hidden_dim, output_dim).to(device)
    dec.load_state_dict(torch.load(decoder_path, map_location=device))
    dec.eval()
    feature_spec = meta.get("feature_spec")
    one_hot_mappings = meta.get("one_hot_mappings", {})
    ord_mappings = meta.get("ord_mappings", {})
    return dec, feature_spec, one_hot_mappings, ord_mappings

def decode_embedding(t: torch.Tensor, feature_spec: dict, one_hot_map: dict, ord_map: dict) -> pd.DataFrame:
    """Decode a single embedding tensor into a pandas DataFrame of features."""
    recon = t.detach().cpu().unsqueeze(0).numpy()
    n_numeric = len([c for c, k in feature_spec.items() if k == "numeric"])
    one_hot_features = [c for c, k in feature_spec.items() if k == "one_hot"]
    n_onehot = sum(len(one_hot_map.get(f, [])) for f in one_hot_features)
    ord_features = [c for c, k in feature_spec.items() if k == "ordinal" or isinstance(k, dict)]

    recon_numeric = recon[:, : n_numeric + n_onehot]
    recon_ordinal = recon[:, n_numeric + n_onehot : n_numeric + n_onehot + len(ord_features)]

    df_numeric = pd.DataFrame(recon_numeric[:, :n_numeric], columns=[c for c, k in feature_spec.items() if k == "numeric"])

    one_hot_array = recon_numeric[:, n_numeric:]
    decoded_onehot = {}
    idx = 0
    for feat in one_hot_features:
        mapping = one_hot_map.get(feat, [])
        width = len(mapping)
        slice_ = one_hot_array[:, idx : idx + width]
        indices = slice_.argmax(axis=1)
        decoded_onehot[feat] = [mapping[i] if i < len(mapping) else None for i in indices]
        idx += width
    df_onehot = pd.DataFrame(decoded_onehot)

    decoded_ord = {}
    for i, feat in enumerate(ord_features):
        spec_val = feature_spec.get(feat)
        if isinstance(spec_val, dict):
            rev_map = {v: k for k, v in spec_val.items()}
        else:
            mapping = ord_map.get(feat, {})
            try:
                rev_map = {int(v): k for k, v in mapping.items()}
            except ValueError:
                rev_map = mapping
        idx_val = int(round(float(recon_ordinal[:, i])))
        decoded_ord[feat] = [rev_map.get(idx_val, "UNK")]
    df_ord = pd.DataFrame(decoded_ord)
    return pd.concat([df_numeric, df_onehot, df_ord], axis=1)

def parse_treatment(treat: str) -> List[int]:
    parts = [int(p) for p in treat.split(",")]
    if len(parts) != 3:
        raise ValueError("Treatment must contain three comma separated integers")
    return parts

def main():
    # For a Jupyter Notebook, preset parameters instead of using argparse
    path = "/home/andrea/Desktop/Sarcoma-DT/Models"
    meta_file =  "saved_models/temporal_cycle_gan/metadata_20250609_063550.json"
    config_file =  "Models/configs/mock_config_cycle_gan.yaml"
    patient_id = 1192056  # replace with a valid patient _id
    treatment = "1,0,0"             # comma separated treatment vector

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    gx, meta = load_temporal_cycle_gan(Path(meta_file), device)

    meta_clin = MetadataHandler(meta["metadata_clinical_file"])
    meta_treat = MetadataHandler(meta["metadata_treatment_file"])

    decoder_treat, feat_spec, oh_map, ord_map = load_decoder(
        meta["metadata_treatment_file"], meta["decoder_treatment"], device
    )

    extractor = MongoExtractor(
        connection_string=os.getenv("MONGO_URI"),
        database_name=os.getenv("MONGO_DB"),
        collection_name=os.getenv("MONGO_COLLECTION"),
        config_file=config_file,
    )

    patient = extractor.get_patient_by_id(patient_id)
    if patient is None:
        raise RuntimeError("Patient not found")

    df = pd.DataFrame([patient])
    dataset = TabularDatasetPID(df, meta_clin, meta_treat, seq_len=meta.get("model_params", {}).get("seq_len", 1))
    loader = DataLoader(dataset, batch_size=1, shuffle=False)
    treat_emb, clin_emb, _ = next(iter(loader))
    clin_emb = clin_emb.to(device)

    treat_vec = parse_treatment(treatment)
    treat_tensor = torch.tensor(treat_vec, dtype=torch.float32, device=device).unsqueeze(0)
    treat_tensor = treat_tensor.unsqueeze(1).repeat(1, clin_emb.size(1), 1)

    with torch.no_grad():
        fake_treat = gx(clin_emb, treat_tensor)

    decoded = [decode_embedding(fake_treat[0, i], feat_spec, oh_map, ord_map) for i in range(fake_treat.size(1))]
    for idx, df_out in enumerate(decoded):
        print(f"\n--- Decoded step {idx} ---")
        print(df_out.to_string(index=False))

# Execute main function directly in the notebook
main()

ModuleNotFoundError: No module named 'models'